In [1]:
import pandas as pd
import requests
import re
import time
import os
from duckduckgo_search import DDGS
from tqdm import tqdm
from bs4 import BeautifulSoup


In [2]:
INPUT_CSV = "Top-1000-EU.csv"
OUTPUT_CSV = "./in_csv/eu/companies_domains_validated.csv"
SLEEP_BETWEEN_QUERIES = 3  # in seconds
MAX_RETRIES = 3
AUTO_SAVE_EVERY = 5  # companies

# --- LOAD COMPANY LIST ---
df_companies = pd.read_csv(INPUT_CSV)
company_names = df_companies['company'].tolist()

In [ ]:
# --- SEARCH DOMAIN USING DUCKDUCKGO ---
def search_domain(company_name):
    query = f"{company_name} official homeland website"
    try:
        with DDGS() as ddgs:
            results = ddgs.text(query)
            for result in results:
                url = result.get("href")
                if url:
                    domain = re.findall(r"https?://(www\.)?([^/]+)", url)
                    if domain:
                        return domain[0][1]
    except Exception as e:
        print(f"[ERROR] Search failed for '{company_name}': {e}")
    return None


# --- VALIDATE DOMAIN EXISTS (HTTPS) ---
def validate_domain(domain):
    try:
        url = f"https://{domain}"
        response = requests.get(url, timeout=5)
        return response.status_code < 400
    except Exception:
        return False

# --- PROCESS SINGLE COMPANY WITH RETRIES ---
def process_company(company_name):
    for attempt in range(1, MAX_RETRIES + 1):
        domain = search_domain(company_name)
        if domain:
            is_valid = validate_domain(domain)
            return {
                "Company": company_name,
                "Domain": domain,
                "Domain_Valid": is_valid
            }
        else:
            print(f"[RETRY] Attempt {attempt} failed for '{company_name}'. Retrying...")
            time.sleep(SLEEP_BETWEEN_QUERIES)
    print(f"[FAILED] Could not resolve domain for '{company_name}' after {MAX_RETRIES} attempts.")
    return {
        "Company": company_name,
        "Domain": "",
        "Domain_Valid": False
    }

# --- IF PARTIAL OUTPUT EXISTS, CONTINUE FROM LAST POINT ---
processed = []
start_index = 0
if os.path.exists(OUTPUT_CSV):
    existing = pd.read_csv(OUTPUT_CSV)
    processed_companies = set(existing['Company'])
    start_index = len(existing)
    processed = existing.to_dict(orient='records')
    print(f"[INFO] Resuming from company #{start_index}...")

# --- MAIN LOOP ---
for idx in tqdm(range(start_index, len(company_names)), desc="Processing companies"):
    company_name = company_names[idx]
    result = process_company(company_name)
    processed.append(result)
    
    # Save after every N companies
    if (idx + 1) % AUTO_SAVE_EVERY == 0 or idx == len(company_names) - 1:
        pd.DataFrame(processed).to_csv(OUTPUT_CSV, index=False)
        print(f"[AUTO-SAVE] Saved progress at {idx + 1} companies.")
    
    time.sleep(SLEEP_BETWEEN_QUERIES)

print(f"\n✅ Process completed. Final results saved to: {OUTPUT_CSV}")


[INFO] Resuming from company #10...


Processing companies:   0%|          | 0/990 [00:00<?, ?it/s]

Processing companies:   0%|          | 1/990 [00:06<1:48:16,  6.57s/it]